# Star Cluster Injection Pipeline - RSP

This notebook provides a configuration-based interface for:
1. Injecting synthetic star clusters into Rubin images
2. Running source detection
3. Computing completeness as a function of magnitude, size, etc.

## Setup

First, clone the pipeline to RSP:
```bash
cd ~
git clone https://github.com/whosneha/INJECT.git
```

In [ ]:
# Setup
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Add pipeline to path
USERNAME = os.environ.get('USER', 'your_username')
PIPELINE_PATH = f'/home/{USERNAME}/INJECT/star-cluster-injection-pipeline'
sys.path.insert(0, PIPELINE_PATH)

# LSST imports
from lsst.daf.butler import Butler

# Pipeline imports
from src.config import InjectionConfig, ClusterConfig, get_template, list_templates
from src.pipeline import InjectionPipeline
from src.retrieval import ClusterRetrieval

print(f"Pipeline loaded from: {PIPELINE_PATH}")
print(f"Available templates: {list_templates()}")

## Configuration

Choose one of the following methods to configure your injection:

### Option A: Use a pre-defined template

In [ ]:
# Option A: Use a template
# Available: 'smooth_plummer', 'smooth_king', 'discrete_kroupa', 'completeness_grid'

config = get_template('smooth_plummer')

# Customize the template
config.run_name = 'my_injection_run'
config.tract = 4431
config.patch = 17
config.band = 'i'
config.n_clusters = 100

print(config)

### Option B: Create a custom configuration

In [ ]:
# Option B: Create a fully custom configuration

config = InjectionConfig(
    # Run identification
    run_name='custom_injection',
    description='Custom star cluster injection run',
    
    # Data source
    repo='dp02',
    collection='2.2i/runs/DP0.2',
    tract=4431,
    patch=17,
    band='i',
    
    # Number of clusters
    n_clusters=200,
    
    # Cluster parameters
    cluster_config=ClusterConfig(
        # Choose method: 'smooth' or 'discrete'
        method='smooth',
        
        # Choose profile: 'plummer', 'king', 'eff', 'sersic'
        profile_type='plummer',
        
        # Magnitude range
        mag_min=19.0,
        mag_max=26.0,
        
        # Size range (pixels)
        r_half_min=2.0,
        r_half_max=40.0,
        
        # For King profile
        concentration_min=20.0,
        concentration_max=100.0,
        
        # For discrete stars
        n_stars_min=50,
        n_stars_max=500,
        imf='kroupa',  # 'kroupa', 'chabrier', 'salpeter'
    ),
    
    # Pipeline settings
    edge_buffer=100,
    add_noise=True,
    use_actual_psf=True,  # Use real Rubin PSF from coadd
    seed=42,
    
    # Output
    output_dir='./injection_output'
)

print("Configuration created!")

### Option C: Load from file

In [ ]:
# Option C: Load from a saved config file
# config = InjectionConfig.from_json('my_config.json')
# config = InjectionConfig.from_yaml('my_config.yaml')

# Save current config for future use
# config.to_json('my_config.json')
# config.to_yaml('my_config.yaml')

## Load Data and Run Pipeline

In [ ]:
# Initialize pipeline
pipeline = InjectionPipeline(config)

# Load data from Butler
print("Loading Butler...")
butler = Butler(config.repo, collections=config.collection)
pipeline.load_data(butler=butler)

print(f"Image shape: {pipeline.image.shape}")

In [ ]:
# Generate injection catalog
catalog = pipeline.generate_catalog()

# Display catalog summary
print(f"\nCatalog Summary:")
print(f"  N clusters: {len(catalog)}")
print(f"  Mag range: [{min(c['magnitude'] for c in catalog):.1f}, {max(c['magnitude'] for c in catalog):.1f}]")
print(f"  r_half range: [{min(c['r_half'] for c in catalog):.1f}, {max(c['r_half'] for c in catalog):.1f}] px")

In [ ]:
# Run injection
injected_image, injection_info = pipeline.run_injection()

In [ ]:
# Visualize injection
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmin, vmax = np.percentile(pipeline.image, [1, 99])

axes[0].imshow(pipeline.image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
axes[0].set_title('Original')

axes[1].imshow(injected_image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
for c in catalog:
    axes[1].scatter(c['x'], c['y'], s=50, facecolors='none', edgecolors='red', lw=1)
axes[1].set_title(f'With {len(catalog)} Injected Clusters')

diff = injected_image - pipeline.image
im = axes[2].imshow(diff, cmap='hot', origin='lower', norm=LogNorm(vmin=0.1, vmax=diff.max()))
axes[2].set_title('Injected Clusters Only')
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()

## Run Detection and Retrieval Analysis

In [ ]:
# Run source detection
# For real analysis, use LSST's detection pipeline and pass the results here
detections = pipeline.run_detection(threshold=3.0)

print(f"Detected {len(detections)} sources")

In [ ]:
# Run retrieval analysis
retrieval = pipeline.run_retrieval(match_radius=5.0)

# Get summary statistics
stats = retrieval.get_summary_statistics()
print("\nRetrieval Summary:")
for key, value in stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.3f}")
    else:
        print(f"  {key}: {value}")

## Completeness Analysis

In [ ]:
# Completeness vs Magnitude
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Magnitude
ax = axes[0]
mag_bins = np.linspace(config.cluster_config.mag_min, config.cluster_config.mag_max, 12)
bin_centers, completeness, completeness_err = retrieval.compute_completeness('magnitude', bins=mag_bins)
ax.errorbar(bin_centers, completeness, yerr=completeness_err, fmt='o-', capsize=3)
ax.axhline(0.5, color='gray', linestyle='--', label='50%')
ax.axhline(0.9, color='gray', linestyle=':', label='90%')
ax.set_xlabel('Magnitude')
ax.set_ylabel('Completeness')
ax.set_title('Completeness vs Magnitude')
ax.set_ylim(0, 1.05)
ax.legend()

# Half-light radius
ax = axes[1]
rh_bins = np.linspace(config.cluster_config.r_half_min, config.cluster_config.r_half_max, 10)
bin_centers, completeness, completeness_err = retrieval.compute_completeness('r_half', bins=rh_bins)
ax.errorbar(bin_centers, completeness, yerr=completeness_err, fmt='o-', capsize=3, color='orange')
ax.axhline(0.5, color='gray', linestyle='--')
ax.set_xlabel('Half-light Radius (pixels)')
ax.set_ylabel('Completeness')
ax.set_title('Completeness vs Size')
ax.set_ylim(0, 1.05)

# 2D completeness map
ax = axes[2]
result_2d = retrieval.compute_completeness_2d('magnitude', 'r_half', n_bins1=8, n_bins2=6)
im = ax.imshow(result_2d['completeness'].T, origin='lower', aspect='auto', cmap='RdYlGn',
               extent=[result_2d['bins1'][0], result_2d['bins1'][-1],
                       result_2d['bins2'][0], result_2d['bins2'][-1]],
               vmin=0, vmax=1)
ax.set_xlabel('Magnitude')
ax.set_ylabel('Half-light Radius (pixels)')
ax.set_title('2D Completeness Map')
plt.colorbar(im, ax=ax, label='Completeness')

plt.tight_layout()
plt.show()

In [ ]:
# Print 50% completeness limits
print("50% Completeness Limits:")
print(f"  Magnitude: {retrieval.get_50_percent_limit('magnitude'):.2f}")
print(f"  Half-light radius: {retrieval.get_50_percent_limit('r_half'):.2f} pixels")

## Save Results

In [ ]:
# Save all results
pipeline.save_results()
print(f"\nResults saved to: {config.output_dir}")

## Quick Reference: Configuration Options

### Cluster Methods
- `'smooth'`: Extended source with analytical profile (fast, good for distant clusters)
- `'discrete'`: Individual stars with IMF and spatial distribution (realistic, resolved clusters)

### Profile Types
- `'plummer'`: Simple analytical profile, good for many clusters
- `'king'`: Tidally truncated, good for globular clusters
- `'eff'`: Power-law profile, good for young clusters
- `'sersic'`: Generalized profile (n=1: exponential, n=4: de Vaucouleurs)

### IMF Options (for discrete stars)
- `'kroupa'`: Kroupa (2001) IMF
- `'chabrier'`: Chabrier (2003) IMF
- `'salpeter'`: Salpeter (1955) IMF